## 1、加载数据集

In [1]:
from datasets import load_dataset

# 加载 CMRC 2018 数据集
datasets = load_dataset("cmrc2018", cache_dir="../data")

# 查看数据集结构
print(datasets)


DatasetDict({
    train: Dataset({
        features: ['id', 'context', 'question', 'answers'],
        num_rows: 10142
    })
    validation: Dataset({
        features: ['id', 'context', 'question', 'answers'],
        num_rows: 3219
    })
    test: Dataset({
        features: ['id', 'context', 'question', 'answers'],
        num_rows: 1002
    })
})


## 2、数据预处理

In [ ]:
from transformers import AutoTokenizer

# 加载预训练的 BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese", cache_dir='./models')

def process_func(datasets):
    tokenized_datasets = tokenizer(
        text=datasets["question"],
        text_pair=datasets["context"],
        return_offsets_mapping=True,
        max_length=384, truncation="only_second", padding="max_length"
    )
    
    offset_mapping = tokenized_datasets.pop("offset_mapping")
    start_positions = []
    end_positions = []
    
    for idx, (offset, answer) in enumerate(zip(offset_mapping, datasets["answers"])):
        # 提取答案的起始和结束字符位置
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])
        
        # 找到上下文的token范围
        sequence_ids = tokenized_datasets.sequence_ids(idx)
        context_start = sequence_ids.index(1)
        context_end = sequence_ids.index(None, context_start) - 1

        # 确定答案是否在上下文中
        if offset[context_end][1] < start_char or offset[context_start][0] > end_char:
            # 答案不在上下文中
            start_positions.append(0)
            end_positions.append(0)
        else:
            # 确定答案的token位置，添加默认值以避免 StopIteration 错误
            start_token_pos = next((i for i in range(context_start, context_end + 1) if offset[i][0] >= start_char), context_start)
            end_token_pos = next((i for i in range(context_end, context_start - 1, -1) if offset[i][1] <= end_char), context_end)
            
            start_positions.append(start_token_pos)
            end_positions.append(end_token_pos)
    
    tokenized_datasets["start_positions"] = start_positions
    tokenized_datasets["end_positions"] = end_positions
    
    return tokenized_datasets


# 对训练集和验证集进行处理
tokenized_datasets = datasets.map(process_func, batched=True, remove_columns=datasets["train"].column_names)


## 3、创建模型

In [3]:
from transformers import BertForQuestionAnswering, TrainingArguments, Trainer

# 加载预训练的 BERT 模型
model = BertForQuestionAnswering.from_pretrained("bert-base-chinese", cache_dir='./models')

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 4、创建评估函数

In [11]:
from cmrc2018_evaluate import calc_f1_score, calc_em_score
import numpy as np
def compute_metrics(eval_pred):
    start_logits, end_logits = eval_pred.predictions
    
    # 通过 argmax 获取预测的开始和结束位置
    predicted_start_positions = np.argmax(start_logits, axis=1)
    predicted_end_positions = np.argmax(end_logits, axis=1)
    
    # 获取真实的 start 和 end 位置
    true_start_positions, true_end_positions = eval_pred.label_ids
    
    # 从 eval_pred.inputs 或 eval_dataset 获取 input_ids
    if eval_pred.inputs is not None:
        input_ids = eval_pred.inputs["input_ids"]
    else:
        # 假设 eval_dataset 可直接获取 tokenized_datasets 的 input_ids
        input_ids = tokenized_datasets["validation"]["input_ids"]
    
    # 解码预测答案和真实答案
    decoded_predictions = []
    decoded_labels = []
    
    for i in range(len(predicted_start_positions)):
        # 解码预测的答案
        pred_ids = input_ids[i][predicted_start_positions[i]:predicted_end_positions[i] + 1]
        predicted_answer = tokenizer.decode(pred_ids, skip_special_tokens=True)
        decoded_predictions.append(predicted_answer)
        
        # 解码真实的答案
        true_ids = input_ids[i][true_start_positions[i]:true_end_positions[i] + 1]
        true_answer = tokenizer.decode(true_ids, skip_special_tokens=True)
        decoded_labels.append(true_answer)
    
    # 使用 CMRC 2018 标准计算 F1 和 EM 分数
    f1_scores = []
    em_scores = []
    
    for prediction, true_answer in zip(decoded_predictions, decoded_labels):
        f1_scores.append(calc_f1_score([true_answer], prediction))
        em_scores.append(calc_em_score([true_answer], prediction))
    
    # 计算平均分数
    avg_f1 = 100.0 * sum(f1_scores) / len(f1_scores)
    avg_em = 100.0 * sum(em_scores) / len(em_scores)
    
    return {"f1": avg_f1, "exact_match": avg_em}


## 5、创建训练器

In [12]:
train_args = TrainingArguments(
    output_dir="./models_for_mrc",      # 输出文件夹
    per_device_train_batch_size=32,  # 训练时的batch_size
    per_device_eval_batch_size=32,   # 验证时的batch_size
    logging_steps=50,                # log 打印的频率
    evaluation_strategy="epoch",     # 评估策略
    save_strategy="epoch",           # 保存策略
    save_total_limit=3,              # 最大保存数
    num_train_epochs=3,              # 训练轮数, 默认为3
    report_to=['tensorboard'],       # tensorboard展示结果
    load_best_model_at_end=True)     # 训练完成后加载最优模型
train_args

d:\program\anaconda3\envs\transformers\Lib\site-packages\transformers\training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
dispatch_batches=None,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
evaluation_s

## 6、开始训练

In [13]:
# 创建 Trainer
trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


In [14]:

# 训练模型
trainer.train()

  0%|          | 0/951 [00:00<?, ?it/s]

{'loss': 1.2619, 'grad_norm': 9.5165376663208, 'learning_rate': 4.737118822292324e-05, 'epoch': 0.16}
{'loss': 0.9056, 'grad_norm': 10.673101425170898, 'learning_rate': 4.474237644584648e-05, 'epoch': 0.32}
{'loss': 0.7695, 'grad_norm': 14.381623268127441, 'learning_rate': 4.211356466876972e-05, 'epoch': 0.47}
{'loss': 0.5681, 'grad_norm': 13.274693489074707, 'learning_rate': 3.9484752891692956e-05, 'epoch': 0.63}
{'loss': 0.609, 'grad_norm': 10.586907386779785, 'learning_rate': 3.6855941114616195e-05, 'epoch': 0.79}
{'loss': 0.59, 'grad_norm': 13.769262313842773, 'learning_rate': 3.4227129337539433e-05, 'epoch': 0.95}


  0%|          | 0/101 [00:00<?, ?it/s]

{'eval_loss': 1.4621779918670654, 'eval_f1': 67.39158225636812, 'eval_exact_match': 52.625038831935385, 'eval_runtime': 30.7685, 'eval_samples_per_second': 104.62, 'eval_steps_per_second': 3.283, 'epoch': 1.0}
{'loss': 0.8433, 'grad_norm': 12.795042991638184, 'learning_rate': 3.159831756046267e-05, 'epoch': 1.1}
{'loss': 0.7311, 'grad_norm': 13.948502540588379, 'learning_rate': 2.8969505783385907e-05, 'epoch': 1.26}
{'loss': 0.7264, 'grad_norm': 16.34007453918457, 'learning_rate': 2.6340694006309152e-05, 'epoch': 1.42}
{'loss': 0.7272, 'grad_norm': 15.428728103637695, 'learning_rate': 2.3711882229232387e-05, 'epoch': 1.58}
{'loss': 0.7185, 'grad_norm': 14.506321907043457, 'learning_rate': 2.1083070452155626e-05, 'epoch': 1.74}
{'loss': 0.7132, 'grad_norm': 14.029378890991211, 'learning_rate': 1.8454258675078864e-05, 'epoch': 1.89}


  0%|          | 0/101 [00:00<?, ?it/s]

{'eval_loss': 1.3582655191421509, 'eval_f1': 69.26961231880581, 'eval_exact_match': 54.39577508543026, 'eval_runtime': 36.3041, 'eval_samples_per_second': 88.668, 'eval_steps_per_second': 2.782, 'epoch': 2.0}
{'loss': 0.6474, 'grad_norm': 11.849699974060059, 'learning_rate': 1.5825446898002103e-05, 'epoch': 2.05}
{'loss': 0.4004, 'grad_norm': 10.234911918640137, 'learning_rate': 1.3196635120925343e-05, 'epoch': 2.21}
{'loss': 0.4058, 'grad_norm': 10.909049034118652, 'learning_rate': 1.056782334384858e-05, 'epoch': 2.37}
{'loss': 0.3941, 'grad_norm': 9.820530891418457, 'learning_rate': 7.93901156677182e-06, 'epoch': 2.52}
{'loss': 0.3971, 'grad_norm': 10.965998649597168, 'learning_rate': 5.310199789695059e-06, 'epoch': 2.68}
{'loss': 0.3979, 'grad_norm': 7.1402082443237305, 'learning_rate': 2.6813880126182968e-06, 'epoch': 2.84}
{'loss': 0.4003, 'grad_norm': 12.547821044921875, 'learning_rate': 5.257623554153523e-08, 'epoch': 3.0}


  0%|          | 0/101 [00:00<?, ?it/s]

{'eval_loss': 1.5228368043899536, 'eval_f1': 70.79273623295575, 'eval_exact_match': 56.72569120844983, 'eval_runtime': 35.5353, 'eval_samples_per_second': 90.586, 'eval_steps_per_second': 2.842, 'epoch': 3.0}
{'train_runtime': 2386.8474, 'train_samples_per_second': 12.747, 'train_steps_per_second': 0.398, 'train_loss': 0.6423912927680713, 'epoch': 3.0}


TrainOutput(global_step=951, training_loss=0.6423912927680713, metrics={'train_runtime': 2386.8474, 'train_samples_per_second': 12.747, 'train_steps_per_second': 0.398, 'total_flos': 5962661340337152.0, 'train_loss': 0.6423912927680713, 'epoch': 3.0})

## 7、评估

In [15]:

# 评估模型
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

  0%|          | 0/101 [00:00<?, ?it/s]

Evaluation results: {'eval_loss': 1.3582655191421509, 'eval_f1': 69.26961231880581, 'eval_exact_match': 54.39577508543026, 'eval_runtime': 30.9797, 'eval_samples_per_second': 103.907, 'eval_steps_per_second': 3.26, 'epoch': 3.0}


## 8、预测

In [16]:
from transformers import pipeline

pipe = pipeline("question-answering", model=model, tokenizer=tokenizer, device=0)
pipe

In [17]:
pipe(question="马云在哪里创建了阿里巴巴？", context="马云在杭州创建了阿里巴巴")

{'score': 0.9876611232757568, 'start': 3, 'end': 5, 'answer': '杭州'}